In [ ]:
import numpy as np
import torch
import json
from munch import Munch
import itertools
from collections import defaultdict
import random
import copy
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import importlib
import time



### Create the HMM

In [ ]:
hidden_states = ['pro','ex1','ex2','int','dis','enh', 'end']
emit_states = ['A','T', 'C', 'G', 'N']
hidden_size, emit_size = len(hidden_states), len(emit_states)

hmm_mat = np.array([
    [.6,.1,.1,.2,0,0,0], #promoter
    [0,.4,.3,.2,.1,0,0], #exon1
    [.0,.3,.4,0,.2,.1,0], #exon2
    [.1,.1,.1,.4,0,.1,.2], #intron
    [0,1/3, 1/3, 0,1/3,0,0], #disease
    [0,.25,.25,.25,0,.25,0], #enhancer
    [0,0,0,0,0,0,1] #end of sequence
])

emit_mat = np.array([ #
    [.1,.1,.4,.4,0], #CG rich promoter
    [.1,.1,.7,.1,0], #Exon 1 favors C
    [.7,.1,.1,.1,0], #Exon 2 favors A
    [.25,.25,.25,.25,0], #Intron 
    [.5,0,.5,0,0], #Disease favors AC
    [.4,.4,.1,.1,0], #AT rich enhancer
    [0,0,0,0,1] #end has no nucleotide emissions
])

init_vec = np.array(
    [.5,0,0,.5,0,0, 0]
)

hmm_transition = {}
for i in range(hidden_size):
    for j in range(hidden_size):
        hmm_transition[hidden_states[i],hidden_states[j]] = hmm_mat[i,j].item()

hmm_emit = {}
for i in range(hidden_size):
    for j in range(emit_size):
        hmm_emit[hidden_states[i],emit_states[j]] = emit_mat[i,j].item()
        
hmm_startprob = {}
for i in range(hidden_size):
    hmm_startprob[hidden_states[i]] = init_vec[i]

end_hmm = copy.deepcopy(Munch(states = hidden_states, emits = emit_states, tprob = hmm_transition, eprob = hmm_emit, initprob = hmm_startprob))

In [ ]:
hidden_states = ['pro','ex1','ex2','int','dis','enh', 'end']
emit_states = ['A','T', 'C', 'G']
hidden_size, emit_size = len(hidden_states), len(emit_states)

hmm_mat = np.array([
    [.6,.1,.1,.2,0,0,0], #promoter
    [0,.4,.3,.2,.1,0,0], #exon1
    [.0,.3,.4,0,.2,.1,0], #exon2
    [.1,.1,.1,.4,0,.1,.2], #intron
    [0,1/3, 1/3, 0,1/3,0,0], #disease
    [0,.25,.25,.25,0,.25,0], #enhancer
    [0,0,0,0,0,0,1] #end of sequence
])

emit_mat = np.array([ #
    [.1,.1,.4,.4], #CG rich promoter
    [.1,.1,.7,.1], #Exon 1 favors C
    [.7,.1,.1,.1], #Exon 2 favors A
    [.25,.25,.25,.25], #Intron 
    [.5,0,.5,0], #Disease favors AC
    [.4,.4,.1,.1], #AT rich enhancer
    [.25,.25,.25,.25] #end has no nucleotide emissions
])

init_vec = np.array(
    [.5,0,0,.5,0,0, 0]
)

hmm_transition = {}
for i in range(hidden_size):
    for j in range(hidden_size):
        hmm_transition[hidden_states[i],hidden_states[j]] = hmm_mat[i,j].item()

hmm_emit = {}
for i in range(hidden_size):
    for j in range(emit_size):
        hmm_emit[hidden_states[i],emit_states[j]] = emit_mat[i,j].item()
        
hmm_startprob = {}
for i in range(hidden_size):
    hmm_startprob[hidden_states[i]] = init_vec[i]

hmm = copy.deepcopy(Munch(states = hidden_states, emits = emit_states, tprob = hmm_transition, eprob = hmm_emit, initprob = hmm_startprob))

### Stay > = 3

In [ ]:
def update_fun(k , r, r_past):
    '''
    r = hidden_states x [1,2,3, 4]
    '''
    prev, count = r_past #r is a tuple
    if k == prev:
        # new_count = min(count + 1 , 3)
        new_count = min(count + 1 , 4)
    else:
        new_count = 1
        
    # if (count == 3 or k == prev) and r == (k, new_count):
    if (count == 4 or k == prev) and r == (k, new_count):
        return True
    else:
        return False

def init_fun(k, r):
    '''
    initial "prob" of r = (m1,m2) from k. is just indicator
    '''

    return r == (k,1)

    
def eval_fun(k, r):
    return 1

m_states = list(itertools.product(hidden_states, list(range(1,4+1))))

stay_cst = Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states)

In [ ]:
update_fun('enh',('enh',1), ('pro',3))

In [ ]:
test_params = create_cst_params(stay_cst, hidden_states)

print(test_params[0].sum(axis = 1),test_params[1].sum(axis = 1),test_params[2].sum(axis = 1))

### Promoter Must Occur in First 10

In [ ]:
def update_fun(k , r, r_past):
    '''
    r = Boolean
    tracks if 'pro' has occured yet or not
    '''        

    return r == (r_past or (k == 'pro')) 

def init_fun(k, r):

    return r == (k == 'pro')

def eval_fun(k,r):
    return 1

m_states = [True,False]

promoter_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

In [ ]:
test_params = create_cst_params(promoter_cst, hidden_states)

print(test_params[0].sum(axis = 1),test_params[1].sum(axis = 1),test_params[2].sum(axis = 1))

## Visit Dis Exactly Once 

In [ ]:
def update_fun(k , r, r_past):
    '''
    r =  [True,False] x [0,1,2]
    r[0] is True if previoius state was NOT dis.
    counts number of entries into dis (ie. number of dis regions)
    '''
    prev, count = r_past

    if prev and k == 'dis':
        count = min(count + 1, 2)    
    
    return r == (k != 'dis',count) 

def init_fun(k, r):

    if k == 'dis':
        return r == (False,1)
    else:
        return r == (True,0)

def eval_fun(k,r):
    return r[1] == 1 #must be exactly 1.

m_states = list(itertools.product([True,False], list(range(3))))


disvisit_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

### Promoter < Disease < Enhancer

In [ ]:
def update_fun(k , r, r_past):
    '''
    r = Boolean_pro x Bool_dis x Bool_enh
    trcks that they occur in sequence
    '''
    occur_pro, occur_dis, occur_enh = r_past
    consist = True
    
    pro_new = (k == 'pro' or occur_pro)
    dis_new = (k == 'dis' or occur_dis)
    enh_new = (k == 'enh' or occur_enh)

    if k == 'dis': #if at anytime dis occurs before pro, that sequence is zero'd out.
        consist = pro_new

    if k == 'enh':
        consist = dis_new 

    return (r == (pro_new, dis_new,enh_new)) and consist

def init_fun(k, r):

    return r == ( k == 'pro', k == 'dis', k == 'enh')

def eval_fun(k,r):
    return 1 

    # return r[2] == True #ensures that enhancer is in the sequence

m_states = list(itertools.product([True, False], repeat=3))

pde_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))



In [ ]:
def update_fun(k , r, r_past):
    '''
    r = Boolean_pro x Bool_dis x Bool_enh
    trcks that they occur in sequence
    '''
    occur_pro, occur_enh = r_past
    consist = True
    
    pro_new = (k == 'pro' or occur_pro)
    enh_new = (k == 'enh' or occur_enh)

    if k == 'enh':
        consist = pro_new 

    return (r == (pro_new,enh_new)) and consist

def init_fun(k, r):

    return r == ( k == 'pro', k == 'enh')

def eval_fun(k,r):
    return 1 #ensures that enhancer is in the sequence

    # return r[1] == True #ensures that enhancer is in the sequence

m_states = list(itertools.product([True, False], repeat=2))

pe_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))



# Hitting Constraint for Dis

In [ ]:
def update_fun(k , r, r_past):
    '''
    r =  [True,False]
    r is a Boolean that tracks if dis has occured
    '''
        
    return r == (r_past or k == 'dis') 

def init_fun(k, r):

    return r == (k == 'dis')

def eval_fun(k,r):
    return r == True #dis has occured

m_states = [True,False]


hitdis_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

# Hitting Constraint for End

In [ ]:
def update_fun(k , r, r_past):
    '''
    r =  [True,False]
    r is a Boolean that tracks if dis has occured
    '''
        
    return r == (r_past or k == 'end') 

def init_fun(k, r):

    return r == (k == 'end')

def eval_fun(k,r):
    return r == True #dis has occured

m_states = [True,False]


hitend_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

# Only Last State is End

In [ ]:
def update_fun(k , r, r_past):
    '''
    r =  True
    r is a dummy var needed for bookeeping in tensor operations
    '''
    counter = r_past
    if k == 'end':
        counter += 1
    return r == min(counter, 2)
    
def init_fun(k, r):

    return r == int(k == 'end')

def eval_fun(k,r):
    return k == 'end' and (r < 2) #have not encountered 'end' before

m_states = [0,1,2]


end_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

### Visit Dis at most Once

In [ ]:
def update_fun(k , r, r_past):
    '''
    r =  [True,False] x [0,1,2]
    r[0] is True if previoius state was NOT dis.
    counts number of entries into dis (ie. number of dis regions)
    '''
    prev, count = r_past

    if prev and k == 'dis':
        count = min(count + 1, 2)    
    
    return r == (k != 'dis',count) 

def init_fun(k, r):

    if k == 'dis':
        return r == (False,1)
    else:
        return r == (True,0)

def eval_fun(k,r):
    return r[1] != 2 #must be 0 or 1

m_states = list(itertools.product([True,False], list(range(3))))



disvisit_atmost_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

# Dummy Constraint

In [ ]:
def update_fun(k, r, r_past):
    return r == True

def init_fun(k, r):

    return r == True

def eval_fun(k, r):
    return 1

m_states = [True,False]

dummy_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))


In [ ]:
dummy_list = [dummy_cst]

In [ ]:
_, dummy_params = convertTensor_list(hmm, dummy_list, dtype = torch.float16, device = 'cpu', hmm_params = None, return_ix = False)

# Experiments: Constrained Inference

In [ ]:
state_list, emit_list = simulation(end_hmm, min_time = 30, stay=4, pro_before=10, ix_list=None, max_attempts=1000)

In [ ]:
cst_list = [promoter_cst,stay_cst,disvisit_cst, pde_cst]

In [ ]:
dummy_list = [dummy_cst]

In [ ]:
opt_state_list, opt_augstateix_list = Viterbi_torch_list(end_hmm, cst_list, emit_list, pro_before = 10, dtype = torch.float32,  device = 'cuda:0', debug = False, num_corr = 0)

In [ ]:
unc_state_list = mv_Viterbi(emit_list, end_hmm, cst = None)

In [ ]:

# Define the common list and its corresponding colors
common_list = ['pro', 'ex1', 'ex2', 'int', 'dis', 'enh', 'end']
colors = {
    'pro': 'blue',
    'ex1': 'green',
    'ex2': 'lime',
    'int': 'yellow',
    'dis': 'red',
    'enh': 'violet',
    'end' : 'black'
}

# Example lists with length 30
# Function to plot a single list as a line with colored segments
tick_length = .75

# def plot_colored_line(ax, data, y_position):
#     x_start = 0
#     for word in data:
#         x_end = x_start + tick_length  # Each word occupies a segment of length 1
#         ax.plot([x_start, x_end], [y_position, y_position], color=colors[word], linewidth=8)
#         x_start = x_end

def plot_colored_line(ax, data, y_position, dash = False):
    x_start = 3.25
    bleed_offset = 0.05  # Adjust this value to account for bleeding

    for word in data:
        x_end = x_start + tick_length  # Each word occupies a segment of length tick_length
        
        # Draw the horizontal colored segment
        ax.plot([x_start + bleed_offset, x_end - bleed_offset], [y_position, y_position], 
                color=colors[word], linewidth=8, solid_capstyle='butt')  # Prevent bleeding
        if dash:
            # Add dashed vertical line at the start of the segment (spanning the entire plot)
            ax.plot([x_start, x_start], [0, 4], color='black', linewidth=0.5, linestyle='--')  # Dashed vertical line
        
        x_start = x_end
    if dash:
    # Add a dashed vertical line at the end of the last segment (spanning the entire plot)
        ax.plot([x_start, x_start], [0, 4], color='black', linewidth=0.5, linestyle='--')  # Dashed vertical line# Function to plot letters on the fourth line
def plot_letters(ax, data, y_position):
    x_start = 3.25
    for letter in data:
        ax.text(x_start + tick_length / 2, y_position, letter,
                fontsize=12, ha='center', va='center')  # Center the letter in the segment
        x_start += tick_length


# Create the plot
fig, ax = plt.subplots(figsize=(9, 3))  # Adjusted figure size for compression

offset = .4
vert = .9
# Plot each list as a line stacked vertically
plot_colored_line(ax, opt_state_list, offset + 3*vert, dash = True) 
plot_colored_line(ax, state_list, offset +2*vert)  
plot_colored_line(ax, unc_state_list, offset  + vert) 

plot_letters(ax, emit_list, offset )  # Line 4 at y=0

fontsize = 10

ax.text(1.25, offset + 3*vert, 'Constrained \nEstimate', va='center', fontsize=fontsize, zorder=8, ha = 'center')
ax.text(1.25, offset + 2*vert, 'Ground Truth', va='center', fontsize=fontsize, zorder=8, ha = 'center')
ax.text(1.2, offset + vert, 'Unconstrained \nEstimate', va='center', fontsize=fontsize, zorder=8, ha = 'center')
ax.text(1.25, offset, 'Nucleotide \nSequence', va='center', fontsize=fontsize, zorder=8, ha = 'center')
# Add legend
legend_patches = [mpatches.Patch(color=color, label=word) for word, color in colors.items()]
ax.legend(handles=legend_patches, loc='center right', title='',bbox_to_anchor=(.99, .5), fontsize=12)

# Adjust plot limits and aesthetics
ax.set_xlim(-1, len(emit_list)) 
ax.set_ylim(0, 4)
ax.set_yticks([]) 
ax.set_xticks([]) 
# ax.set_title('Functional Region Inference: Constrained vs. Unconstrained', fontsize=12)
ax.grid(False)

ax.set_ylim(0, 3.5)  # Reduce the upper limit from 4 to 3.5
# Show the plot
plt.tight_layout()

# plt.savefig('dna_inference.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


# Satisfaction Time Algorithm

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
import matplotlib.patches as mpatches

state_list, emit_list = simulation(end_hmm, min_time=20, stay=4, pro_before=10, ix_list=None, max_attempts=1000)
sat_cst_list = [promoter_cst, stay_cst, pde_cst, hitdis_cst]
sat_probs, alpha, beta = satTime_torch_list(
    end_hmm,
    sat_cst_list,
    emit_list,
    pro_before=10,
    dtype=torch.float32,
    device='cpu',
    debug=False
)

common_list = ['pro', 'ex1', 'ex2', 'int', 'dis', 'enh', 'end']
colors = {
    'pro': 'blue',
    'ex1': 'green',
    'ex2': 'lime',
    'int': 'yellow',
    'dis': 'red',
    'enh': 'violet',
    'end': 'black'
}

fig, ax = plt.subplots(figsize=(12, 4))
indices = list(range(len(state_list)))

norm_probs = sat_probs / sat_probs.sum()
ax.plot(indices, norm_probs.tolist(), marker='o')

# Use data coordinates in x, axes coordinates in y
trans = blended_transform_factory(ax.transData, ax.transAxes)

tick_length = 1
state_y = -0.06
nuc_y = -0.14

# Draw colored hidden-state bar outside the plot
x_start = -0.5
for i, state in enumerate(state_list):
    x_end = x_start + tick_length

    ax.plot(
        [x_start, x_end],
        [state_y, state_y],
        color=colors[state],
        linewidth=12,
        solid_capstyle='butt',
        clip_on=False,
        transform=trans
    )

    x_start = x_end

# Draw nucleotide sequence outside the plot
for i, nuc in enumerate(emit_list):
    ax.text(
        i,
        nuc_y,
        nuc,
        fontsize=11,
        ha='center',
        va='center',
        clip_on=False,
        transform=trans
    )

# Axis labels
ax.set_xlabel('Ground Truth Functional Regions', fontsize=12, labelpad=30)
ax.set_ylabel('Probability', fontsize=12)

# Hide x tick labels since sequence shown below
ax.set_xticks(indices)
ax.set_xticklabels([''] * len(indices))

ax.set_ylim([-.1,1])
# Highlight dis region
start = None
end = None
for ix, state in enumerate(state_list):
    if state == "dis":
        if start is None:
            start = ix
        end = ix

if start is not None and end is not None:
    ax.axvspan(start, end, color='red', alpha=0.3)

# Legend outside plot
legend_patches = [mpatches.Patch(color=color, label=word) for word, color in colors.items()]
ax.legend(
    handles=legend_patches,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    fontsize=10,
    borderaxespad=0.
)

ax.grid(True)

# Leave room at bottom for outside annotations and at right for legend
plt.subplots_adjust(bottom=0.32, right=0.82)

plt.show()

# Fixed Length Sampling

In [ ]:
sampled_path = ffbs_torch_list(end_hmm, cst_list, emit_list, pro_before = 10, dtype = torch.float32,  device = 'cuda:0', debug = False)

# Variable Length Sampling

In [ ]:
state_list, emit_list = simulation(end_hmm, min_time = 20, stay=4, pro_before=10, ix_list=None, max_attempts=1000)

In [ ]:
sat_cst_list = [promoter_cst,stay_cst,pde_cst, hitdis_cst] #omit the disvisit constraint

In [ ]:
sat_probs, alpha, beta = satTime_torch_list(end_hmm, sat_cst_list, emit_list, pro_before = 10, dtype = torch.float32,  device = 'cpu', debug = False)

In [ ]:
emit_dict = {2:'C', 4:'C', 5:'T', 20:'A', 29: 'N'}
samplecst_list = cst_list + [end_cst]
sample_path = ffbs_torch_list(end_hmm, samplecst_list, length_param = (30,emit_dict), pro_before = 10, dtype = torch.float32,  device = 'cuda:0', debug = False)

In [ ]:
timecst_list = cst_list + [hitend_cst]
emit_dict2 = {2:'C', 4:'C', 5:'T', 20:'A'}
sat_probs, alpha, beta = satTime_torch_list(end_hmm, timecst_list, (50,emit_dict2), pro_before = 10, dtype = torch.float32,  device = 'cpu', debug = False)

In [ ]:
sample_path, length = variable_length_sampling(end_hmm, timecst_list, samplecst_list, (50,emit_dict2), min_time = 20, pro_before = 10, dtype = torch.float32,  device = 'cpu', debug = False)


In [ ]:
# State colors
colors = {
    'pro': 'blue',
    'ex1': 'green',
    'ex2': 'lime',
    'int': 'yellow',
    'dis': 'red',
    'enh': 'violet',
    'end': 'black'
}

tick_length = 0.75


def plot_colored_line(ax, seq, y, colors, y_min, y_max, tick_length=0.75):
    bleed = 0.05

    for i, state in enumerate(seq):
        x0 = i * tick_length
        x1 = x0 + tick_length

        # colored horizontal segment
        ax.plot(
            [x0 + bleed, x1 - bleed],
            [y, y],
            color=colors[state],
            linewidth=8,
            solid_capstyle='butt'
        )

        # dashed vertical boundary at left edge of segment
        ax.plot(
            [x0, x0],
            [y_min, y_max],
            color='black',
            linewidth=0.5,
            linestyle='--'
        )

    # dashed vertical boundary at right edge of final segment
    if len(seq) > 0:
        x_end = len(seq) * tick_length
        ax.plot(
            [x_end, x_end],
            [y_min, y_max],
            color='black',
            linewidth=0.5,
            linestyle='--'
        )


def plot_partial_observations(ax, obs_dict, y, tick_length=0.75):
    for idx in sorted(obs_dict):
        x = idx * tick_length + tick_length / 2
        ax.text(x, y, obs_dict[idx], ha='center', va='center', fontsize=12)


def plot_sequences(sequences, obs_dict, colors, tick_length=0.75, figsize=(10, 4)):
    """
    sequences: list of sequences, where each sequence is a list of state strings
    obs_dict: dict like {10: 'C', 15: 'A'}
    """
    fig, ax = plt.subplots(figsize=figsize)

    offset = 0.4
    vert = 0.9
    n = len(sequences)

    y_min = 0
    y_max = offset + (n + 1.0) * vert

    # plot each sequence
    for j, seq in enumerate(sequences):
        y = offset + (n - j) * vert
        plot_colored_line(ax, seq, y, colors, y_min, y_max, tick_length=tick_length)

    # plot observed nucleotides on bottom row
    plot_partial_observations(ax, obs_dict, offset, tick_length=tick_length)

    # legend outside the plot
    legend_patches = [mpatches.Patch(color=c, label=s) for s, c in colors.items()]
    ax.legend(
        handles=legend_patches,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        fontsize=10,
        borderaxespad=0.
    )

    # axis limits
    max_seq_len = max((len(seq) for seq in sequences), default=0)
    max_obs_idx = max(obs_dict.keys(), default=-1) + 1
    max_len = max(max_seq_len, max_obs_idx)

    ax.set_xlim(-0.5, max_len * tick_length + 0.5)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title('Posterior Sampling of Constrained DNA Sequences', fontsize=  18)
    ax.set_xlabel('Partially Observed Nucelotide Sequence')

    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()


In [ ]:
n_samples = 5
sample_list = []
for i in range(n_samples):
    sample_path, length = variable_length_sampling(end_hmm, timecst_list, samplecst_list, (50,emit_dict2), min_time = 20, pro_before = 10, dtype = torch.float32,  device = 'cpu', debug = False)
    sample_list.append(sample_path)

In [ ]:
plot_sequences(sample_list, emit_dict2, colors)


# Satisifaction Probability Estimation

In [ ]:
from dna_algorithms import create_cst_params, convertTensor_list, simulation, Viterbi_torch_list, \
satTime_torch_list, ffbs_torch_list, variable_length_sampling, satprob_torch_list
from mv_Viterbi import mv_Viterbi

In [ ]:
sat_cst_list = [promoter_cst,stay_cst,pde_cst, hitdis_cst]

In [ ]:
probs = satprob_torch_list(end_hmm, sat_cst_list, emit_list, pro_before = 10, dtype = torch.float32,  device = 'cpu', debug = False, num_corr = 0)

In [ ]:
probs

# Doesn't work for length 30

In [ ]:
B = 500
probsat_cst_list = [promoter_cst,stay_cst, pe_cst, hitdis_cst]
prob_dis = []
for b in range(B):
    state_list, emit_list = simulation(end_hmm, stay=4, pro_before=10, ix_list=None, max_attempts=1000)
    probs = satprob_torch_list(end_hmm, probsat_cst_list, emit_list, pro_before = 10, dtype = torch.float32,  device = 'cuda:0', debug = False, num_corr = 0)
    prob_dis.append(probs[0].item())

In [ ]:
np.array(prob_dis).mean()

In [ ]:
bins = np.linspace(0, 1, 21) 

plt.figure(figsize=(6, 3))

plt.hist(prob_dis, bins=bins, edgecolor='black')

# Adding titles and labels
# plt.title('Histogram of Probabilities')
plt.xlabel('Posterior Probability of Having Mutation')
plt.ylabel('Counts')

# Show the plot
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

x = np.linspace(0, 1, 200)
kde = gaussian_kde(prob_dis)

plt.figure(figsize=(6, 3))
plt.plot(x, kde(x))

plt.title('Distribution of Mutation Probability Estimates (n=500)')
plt.xlabel('Posterior Mutation Probability')
# plt.ylabel('Density')
plt.yticks([])
plt.xlim(0, 1)
plt.show()
